In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from tqdm import tqdm

import importlib

import w1_tree as wt

import plotly.express as px

PATH = "../data_clean/"
# df_topics=pd.read_csv(PATH+'Floriana_topic_mapping.csv')

In [43]:
mode_list = ["individual", "collaborative"]

In [44]:
df_cs = pd.read_csv(PATH+"df_country_subfield_roll_5y.csv", index_col=[0, 1, 2]).dropna(how="all")
df_cs.columns = df_cs.columns.astype(int)

year_list = df_cs.index.get_level_values(1).unique().sort_values().to_list()
country_list = df_cs.index.get_level_values(0).unique().sort_values().to_list()

df_prob = df_cs.div(df_cs.sum(axis=1), axis=0)

In [45]:
df_subfield = (
    pd.read_csv(PATH+"df_topics.csv")
    [["subfield_id", "subfield_name", "field_id", "field_name", "domain_id", "domain_name"]]
    .drop_duplicates()
    .sort_values(["domain_id", "field_id", "subfield_id"])
)

w1_tree = wt.W1Tree(df_subfield[["domain_id", "field_id", "subfield_id"]])

### Country A vs Country B (every country, every year, two modes)

In [98]:
df_country_dist_list = []
for year_loop in tqdm(year_list):
    for mode_loop in mode_list:
        df_tmp = df_prob.xs((year_loop, mode_loop), level=(1, 2))
        df_country_dist_list.append(w1_tree.dist(df_tmp, df_tmp).assign(year=year_loop, mode=mode_loop).reset_index(names=["country"]))
df_country_dist = pd.concat(df_country_dist_list).set_index(["country", "year", "mode"]).xs("distance", axis=1)

100%|██████████| 50/50 [00:05<00:00,  8.60it/s]


In [150]:
df_country_dist.to_csv(PATH+"df_dist_country.csv")

In [35]:
df_country_dist = pd.read_csv(PATH+"df_dist_country.csv", index_col=[0, 1, 2])

In [39]:
df_country_dist.xs("individual", level=2).xs(2023, level=1)[df_country_dist.xs("individual", level=2).xs(2023, level=1).index.to_list()]

,AE,AF,AG,AL,AM,AO,AR,AT,AU,AZ,...,UZ,VA,VE,VG,VN,WS,YE,ZA,ZM,ZW
country,,,,,,,,,,,,,,,,,,,,,
AE,0.000000,0.434038,0.593298,0.321629,0.166399,0.404730,0.289113,0.114038,0.217829,0.143394,...,0.266301,0.738410,0.288345,0.551111,0.086079,0.738784,0.149754,0.184332,0.329959,0.355183
AF,0.434038,0.000000,0.648299,0.138607,0.352805,0.211802,0.219977,0.401139,0.408355,0.335932,...,0.237002,0.334292,0.194536,0.900162,0.438866,0.335169,0.426628,0.292131,0.204036,0.146356
AG,0.593298,0.648299,0.000000,0.609861,0.698907,0.483618,0.621948,0.554498,0.455710,0.672674,...,0.673874,0.774775,0.595778,1.000000,0.661312,0.774775,0.473707,0.590172,0.511242,0.666906
AL,0.321629,0.138607,0.609861,0.000000,0.296479,0.167344,0.136589,0.301067,0.306105,0.279268,...,0.188556,0.436719,0.104200,0.819447,0.377954,0.437260,0.329153,0.191606,0.125849,0.095432
AM,0.166399,0.352805,0.698907,0.296479,0.000000,0.399916,0.225733,0.181777,0.289022,0.042429,...,0.151836,0.624502,0.254718,0.591714,0.144352,0.624839,0.249493,0.156759,0.325529,0.253658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WS,0.738784,0.335169,0.774775,0.437260,0.624839,0.467093,0.539355,0.718696,0.723107,0.637400,...,0.535658,0.009009,0.518752,1.000000,0.748788,0.000000,0.745364,0.614603,0.512979,0.450472
YE,0.149754,0.426628,0.473707,0.329153,0.249493,0.297847,0.285721,0.103097,0.096644,0.220507,...,0.263564,0.744931,0.281831,0.683774,0.208276,0.745364,0.000000,0.180340,0.254270,0.351241
ZA,0.184332,0.292131,0.590172,0.191606,0.156759,0.276918,0.113378,0.119185,0.152320,0.143251,...,0.102259,0.614020,0.115309,0.721397,0.226478,0.614603,0.180340,0.000000,0.194257,0.181795


### Year vs (Year + 1) (every country, every year, two modes)

In [146]:
df_years_dist_list = []
for country_loop in tqdm(country_list):
    for mode_loop in mode_list:
        df_tmp = df_prob.xs((country_loop, mode_loop), level=(0, 2))
        df_years_dist_list.append(w1_tree.dist(df_tmp, df_tmp).assign(country=country_loop, mode=mode_loop).reset_index(names=["year_from"]))

df_years_dist = (
    pd.concat(df_years_dist_list)
    .set_index(["country", "year_from", "mode"])
    .xs("distance", axis=1)
    .reset_index()
    .melt(id_vars=["country", "year_from", "mode"], var_name="year_to", value_name="distance")
    .set_index(["country", "mode"])
    .sort_values(["country", "mode", "year_from", "year_to"])
    .dropna(subset="distance")
    .query("year_from != year_to")
)

100%|██████████| 209/209 [00:01<00:00, 105.15it/s]


In [151]:
df_years_dist.to_csv(PATH+"df_dist_year.csv")

In [4]:
df_years_dist = pd.read_csv(PATH+"df_dist_year.csv")

### Domestic vs Collaborative (every country, every year)

In [56]:
"NG" in country_list

True

In [59]:
df_cs.xs(("NG", 1974), level=(0, 1))

,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,1212,2104,1409,2922,1410,2402,2714,1200,2718,3002
mode,,,,,,,,,,,,,,,,,,,,,
individual,NaN,18.0,23.0,10.0,49.0,27.0,NaN,13.0,15.0,113.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
df_prob.xs(("NG", 1974), level=(0, 1))

,1100,1102,1103,1104,1105,1106,1107,1108,1109,1110,...,1212,2104,1409,2922,1410,2402,2714,1200,2718,3002
mode,,,,,,,,,,,,,,,,,,,,,
individual,NaN,0.016408,0.020966,0.009116,0.044667,0.024613,NaN,0.011851,0.013674,0.103008,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
df_mode_dist_list = []
for country_loop in country_list:
    for year_loop in year_list:
        df_tmp = df_prob.xs((country_loop, year_loop), level=(0, 1))
        df_dist_tmp = w1_tree.dist(df_tmp, df_tmp).assign(year=year_loop, country=country_loop)
        if not df_dist_tmp.empty:
            df_mode_dist_list.append(df_dist_tmp)

mode_symmetry = "individual"

df_mode_dist = (
    pd.concat(df_mode_dist_list)
    .reset_index(names="mode_from")
    .set_index(["country", "year", "mode_from"])
    .reset_index()
    .melt(id_vars=["country", "year", "mode_from"], var_name="mode_to", value_name="distance")
    .query("(mode_from != mode_to) and (mode_from == @mode_symmetry)")
    .drop(columns=["mode_from", "mode_to"])
    .set_index(["country", "year"])
    .dropna(subset="distance")
)

In [54]:
df_mode_dist.to_csv(PATH+"df_dist_mode.csv")